# Session 3: Can It Survive Regime Shifts? HMM Backtesting and Bandit Challengers

We have a Cobb-Douglas rebalancing engine. It adapts to sentiment, enforces safety rules, and outperforms static portfolios on smooth synthetic data. But the real question is: _can it survive a regime shift?_ In this session, we use Hidden Markov Models with Poisson jumps ([JumpHMM.jl](https://github.com/varnerlab/JumpHMM.jl)) to generate synthetic market paths that include normal markets, stressed markets, and sudden crashes — then we run the engine across hundreds of these paths to see if it holds up.

We also introduce a powerful **challenger**: a **multi-armed bandit** that adaptively _learns_ which assets to include in the portfolio. Can an adaptive learner beat a utility-maximizing allocator when markets change regime?

> **By the end of this session, you will be able to:**
> * Explain how Hidden Markov Models with Poisson jumps generate realistic regime-switching synthetic data
> * Fit and tune a JumpHMM model from price data and generate Monte Carlo test paths
> * Backtest the Session 2 Cobb-Douglas rebalancing engine across hundreds of HMM-generated scenarios
> * Implement a combinatorial bandit for adaptive asset selection and compare it to the utility-based allocator
> * Produce a formal validation report with pass/fail criteria for deployment readiness

Let's stress-test this engine for real.

## Examples
The following example notebooks accompany this lecture:

\u27a1 [Example: HMM-Powered Backtesting](eCornell-AI-Finance-L3-Example-HMMBacktesting-May-2026.ipynb). In this example, we fit a JumpHMM model to synthetic training data, generate regime-switching test paths, and backtest the Cobb-Douglas rebalancing engine across hundreds of Monte Carlo scenarios.

\u27a1 [Example: Bandit Challenger and Validation Report](eCornell-AI-Finance-L3-Example-BanditChallengerValidation-May-2026.ipynb). In this example, we implement the epsilon-greedy bandit for adaptive asset selection, backtest it against the Cobb-Douglas engine and buy-and-hold, and produce a formal validation report with pass/fail criteria.

## Why GBM Isn't Enough: The Case for Regime-Switching Models

In Sessions 1 and 2, we generated synthetic data using geometric Brownian motion (GBM) — returns drawn from a single multivariate normal distribution with constant drift and volatility. This is fine for prototyping, but it misses three features of real markets that matter most for stress testing:

> **1. Volatility clustering.** Real markets exhibit periods of high volatility followed by more high volatility (and vice versa). GBM assumes constant volatility — every day looks the same statistically.
>
> **2. Fat tails.** Real returns have more extreme events than a normal distribution predicts. The 2008 crash, the March 2020 COVID drop, and the April 2025 tariff shock were all multi-sigma events under GBM assumptions.
>
> **3. Regime persistence.** Markets don't randomly switch between calm and crisis — they tend to _stay_ in a regime for extended periods. A bear market lasts months, not minutes. GBM has no concept of "regimes."

Hidden Markov Models address all three: they model the market as switching between hidden states (regimes), each with its own return distribution, with persistent transitions between states.

## The JumpHMM: Hidden Markov Models with Poisson Jumps

The [JumpHMM.jl](https://github.com/varnerlab/JumpHMM.jl) package implements a hybrid Hidden Markov Model with a Poisson jump mechanism, following [Alswaidan and Varner (2025)](https://arxiv.org/abs/2603.10202). The model has four components:

> **1. State Space.** The continuous space of excess log-growth rates $G_t = \frac{1}{\Delta t}\ln\frac{P_t}{P_{t-1}} - r_f$ is discretized into $N$ states using equal-probability quantile bins of a fitted Laplace distribution.

> **2. Transition Matrix.** An $N \times N$ row-stochastic matrix $\mathbf{T}$ governs the probability of moving between states: $P(S_t = j \mid S_{t-1} = i) = T_{ij}$. This captures regime persistence — the diagonal elements represent the probability of _staying_ in the current state.

> **3. Emission Distributions.** Each state $k$ has its own Student-t emission: $G_t \mid S_t = k \sim \mu_k + \sigma_k \cdot t_\nu$. The Student-t (rather than Gaussian) captures fat tails within each state.

> **4. Poisson Jump Mechanism.** At each time step, with probability $\epsilon$, a "jump event" fires. The duration of the jump $K \sim \text{Poisson}(\lambda)$ determines how many consecutive steps are forced into tail states. This produces realistic **volatility clustering** — multiple consecutive extreme-return days during crises.

### Fitting Workflow

```
prices \u2192 excess_growth_rates \u2192 fit(LaplacePartition) \u2192 assign_states 
       \u2192 estimate_transition \u2192 fit_emissions \u2192 tune(\u03b5, \u03bb) \u2192 JumpHiddenMarkovModel
```

The `tune` step uses a grid search over $(\epsilon, \lambda)$ to match the empirical kurtosis and autocorrelation function of absolute returns — the two key signatures of realistic market behavior that GBM misses.

## Backtesting Framework: Monte Carlo over Regime-Switching Paths

With a fitted JumpHMM, we can generate _hundreds_ of plausible future market paths — each with its own pattern of calm periods, volatility spikes, and crash events. The backtesting framework runs each strategy across all paths and collects the distribution of outcomes:

> **Backtesting Pseudo-code:**
> 1. Fit JumpHMM to training data; tune jump parameters.
> 2. Generate $M$ synthetic market paths of length $T$ (e.g., $M = 100$ paths, $T = 336$ days).
> 3. For each path, generate per-ticker prices via SIM from the market index.
> 4. For each path, run each strategy (Cobb-Douglas engine, bandit, buy-and-hold) and record: final wealth, max drawdown, Sharpe ratio.
> 5. Aggregate: compute medians, percentiles, and failure rates across all $M$ paths.

This is _not_ a single backtest on a single path — it's a Monte Carlo simulation that reveals the _distribution_ of possible outcomes.

## The Bandit Challenger: Adaptive Asset Selection

The Cobb-Douglas engine from Session 2 uses _all_ assets and lets the preference weights $\gamma_i$ determine allocation. But what if some assets should be **excluded entirely** during certain regimes? This is the _asset selection_ problem, and it's naturally framed as a **multi-armed bandit**.

### Combinatorial Bandit Formulation

With $K$ assets, there are $2^K - 1$ possible non-empty subsets (arms). Each arm $a$ is a binary vector: $a_i = 1$ means "include asset $i$", $a_i = 0$ means "exclude it."

> **The world function.** Given an action vector $a$, the Cobb-Douglas allocator computes the optimal allocation over the _included_ assets and returns the utility as the reward:
>
> $$r(a) = U_{\text{CD}}(n_1^\star(a), \ldots, n_K^\star(a))$$
>
> Excluded assets are forced to the minimum $\epsilon$ position.

### Epsilon-Greedy with Decaying Exploration

We use an **epsilon-greedy** policy where the exploration rate decays over time:

> $$\epsilon_t = t^{-1/3} \cdot (K \cdot \ln t)^{1/3}$$
>
> At each round:
> * With probability $\epsilon_t$: **explore** — pull a random arm
> * With probability $1 - \epsilon_t$: **exploit** — pull the arm with the highest average reward

The reward estimates are updated via a running average:

> $$\hat{\mu}_a \leftarrow \hat{\mu}_a + \alpha \cdot (r_t - \hat{\mu}_a)$$

### Why Bandits as a Challenger?

The Cobb-Douglas engine is a **fixed rule**: given sentiment and SIM parameters, it always computes the same allocation. The bandit is an **adaptive learner**: it explores different asset subsets and learns which ones produce the highest utility. This is a fundamentally different approach:

| | Cobb-Douglas Engine | Bandit Selector |
|---|---|---|
| **Decides** | How much of each asset | Which assets to include |
| **Adapts via** | Sentiment signal $\lambda_t$ | Trial-and-error learning |
| **Strength** | Fast, analytical, interpretable | Can discover non-obvious subsets |
| **Weakness** | Always uses all assets | Needs iterations to converge |

In Session 4, we'll combine them: the bandit selects assets, then the Cobb-Douglas allocator decides how much of each. But first, let's see how they compare head-to-head.

## Validation Criteria: Pass/Fail for Deployment

The final output of this session is a **validation report** — a formal document that says whether each strategy is ready for production (Session 4) or needs more work:

| Criterion | Threshold | Rationale |
|-----------|----------|----------|
| **Median Sharpe $\geq$ 0.3** | Must beat risk-free on a risk-adjusted basis | If Sharpe < 0.3, the strategy isn't generating enough return per unit of risk |
| **Median Max Drawdown $\leq$ 25%** | Worst-case loss must be survivable | Drawdowns > 25% trigger investor redemptions and regulatory scrutiny |
| **Failure Rate $\leq$ 10%** | No more than 10% of paths end below starting capital | A strategy that loses money >10% of the time is unreliable |
| **Beats Buy-and-Hold** | Median wealth must exceed passive benchmark | If the strategy can't beat passive investing, the complexity isn't justified |

> **What does "pass" mean?** A strategy that passes all four criteria across HMM-generated paths has demonstrated robustness to regime shifts, fat tails, and volatility clustering. It is _not_ guaranteed to work in production — real markets have additional risks — but it has cleared the minimum bar for deployment consideration.

## Summary

> **Key Takeaways from Session 3:**
> * GBM-generated synthetic data misses volatility clustering, fat tails, and regime persistence — all critical for realistic stress testing
> * JumpHMM combines a Hidden Markov Model with Poisson jumps to produce synthetic paths that match real market statistical signatures
> * Monte Carlo backtesting across hundreds of paths reveals the _distribution_ of outcomes, not just a single case
> * The **combinatorial bandit** provides an adaptive challenger — it learns which asset subsets produce the highest Cobb-Douglas utility
> * Formal validation criteria (Sharpe, drawdown, failure rate) determine whether strategies are ready for production

**Next Session:** In [Session 4](../session-4/L4/eCornell-AI-Finance-L4-Lecture-ProductionOps-May-2026.ipynb), strategies that pass validation move to production. The bandit and Cobb-Douglas allocator will work _together_ — the bandit selects assets continuously, and the allocator decides how much of each. We add sentiment monitoring, alert systems, and human override protocols.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The models and strategies described are pedagogical tools using synthetic data. Real-world deployment requires additional regulatory, legal, and operational considerations.